In [8]:
# ==============================
# PROJECT SETUP
# ==============================

import requests
import pandas as pd
import json
import time
import os

# OpenAQ API
API_KEY = "fc3225065e4850225ad568f999814ee9408557fe05db6dc429a98164c7df439f"

LOCATION_ID = 10907

HEADERS = {
    "X-API-Key": API_KEY
}

# Gwalior sensors
usable_sensors = {
    "pm25": 12236153,
    "pm10": 12236152,
    "no2": 12236150,
    "co": 12236148,
    "so2": 12236155,
    "o3": 12236151,
    "temperature": 12236156,
    "humidity": 12236154
}

# Google Drive is NOT needed now.
# We will save data locally in this Jupyter project.

print("Project setup loaded successfully!")
print("Sensors:", len(usable_sensors))

Project setup loaded successfully!
Sensors: 8


In [1]:
import requests
import pandas as pd

print("Requests version:", requests.__version__)
print("Pandas version:", pd.__version__)
print("Environment is ready!")


Requests version: 2.32.3
Pandas version: 2.2.3
Environment is ready!


In [2]:
API_KEY = "fc3225065e4850225ad568f999814ee9408557fe05db6dc429a98164c7df439f"

print("API key loaded successfully!")

API key loaded successfully!


In [3]:
LOCATION_ID = 10907

URL = f"https://api.openaq.org/v3/locations/{LOCATION_ID}/latest"

HEADERS = {
    "X-API-Key": 'fc3225065e4850225ad568f999814ee9408557fe05db6dc429a98164c7df439f'
}

print("Gwalior station configured!")
print("Location ID:", LOCATION_ID)

Gwalior station configured!
Location ID: 10907


In [9]:
response = requests.get(
    URL,
    headers=HEADERS
)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("OpenAQ connection successful!")
else:
    print(response.text)

Status Code: 200
OpenAQ connection successful!


In [6]:
def get_sensor_measurements_safe(sensor_id, start_date, end_date):

    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"

    all_results = []
    page = 1

    while True:

        params = {
            "datetime_from": start_date,
            "datetime_to": end_date,
            "limit": 1000,
            "page": page
        }

        for attempt in range(3):

            try:
                response = requests.get(
                    url,
                    headers=HEADERS,
                    params=params,
                    timeout=60
                )

                if response.status_code == 200:
                    break

                print(
                    f"Page {page}: HTTP {response.status_code} "
                    f"(attempt {attempt + 1}/3)"
                )

            except requests.exceptions.RequestException:
                print(
                    f"Page {page}: connection error "
                    f"(attempt {attempt + 1}/3)"
                )

            time.sleep(3)

        else:
            print("Failed after 3 attempts.")
            break

        data = response.json()
        results = data["results"]

        if not results:
            break

        all_results.extend(results)

        print(f"Page {page}: {len(results)} readings")

        if len(results) < 1000:
            break

        page += 1

    return all_results

In [10]:
feb_pm25 = get_sensor_measurements_safe(
    sensor_id=12236153,
    start_date="2025-02-19T00:00:00+05:30",
    end_date="2025-03-01T00:00:00+05:30"
)

print("Total PM2.5 readings:", len(feb_pm25))

Page 1: 910 readings
Total PM2.5 readings: 910


In [11]:
feb_data = {}

for parameter, sensor_id in usable_sensors.items():

    print("\n" + "=" * 50)
    print("Collecting:", parameter)
    print("=" * 50)

    results = get_sensor_measurements_safe(
        sensor_id=sensor_id,
        start_date="2025-02-19T00:00:00+05:30",
        end_date="2025-03-01T00:00:00+05:30"
    )

    feb_data[parameter] = results

    print("Total:", len(results))


Collecting: pm25
Page 1: 910 readings
Total: 910

Collecting: pm10
Page 1: 854 readings
Total: 854

Collecting: no2
Page 1: 884 readings
Total: 884

Collecting: co
Page 1: 868 readings
Total: 868

Collecting: so2
Page 1: 860 readings
Total: 860

Collecting: o3
Page 1: 883 readings
Total: 883

Collecting: temperature
Page 1: 912 readings
Total: 912

Collecting: humidity
Page 1: 912 readings
Total: 912


In [12]:
raw_folder = "raw_data"

os.makedirs(raw_folder, exist_ok=True)

print("Raw data folder ready:", os.path.abspath(raw_folder))

Raw data folder ready: C:\Users\LOQ\Gwalior AQI Project\01_Data_Collection\raw_data


In [13]:
for parameter, results in feb_data.items():

    df = pd.DataFrame(results)

    filename = f"{raw_folder}/2025-02_{parameter}.csv"

    df.to_csv(filename, index=False)

    print(parameter, "saved →", filename)

pm25 saved → raw_data/2025-02_pm25.csv
pm10 saved → raw_data/2025-02_pm10.csv
no2 saved → raw_data/2025-02_no2.csv
co saved → raw_data/2025-02_co.csv
so2 saved → raw_data/2025-02_so2.csv
o3 saved → raw_data/2025-02_o3.csv
temperature saved → raw_data/2025-02_temperature.csv
humidity saved → raw_data/2025-02_humidity.csv


In [14]:
feb_tables = []

for parameter in usable_sensors.keys():

    filename = f"{raw_folder}/2025-02_{parameter}.csv"

    df = pd.read_csv(filename)

    # Extract the measurement timestamp
    df["timestamp"] = pd.to_datetime(
        df["period"].apply(
            lambda x: eval(x)["datetimeFrom"]["local"]
        )
    )

    # Keep only what we need
    df = df[["timestamp", "value"]]

    # Rename the value column
    df = df.rename(columns={
        "value": parameter
    })

    feb_tables.append(df)

# Combine all parameters using timestamp
feb_clean = feb_tables[0]

for df in feb_tables[1:]:
    feb_clean = pd.merge(
        feb_clean,
        df,
        on="timestamp",
        how="outer"
    )

# Sort chronologically
feb_clean = feb_clean.sort_values("timestamp").reset_index(drop=True)

feb_clean.head(20)

,timestamp,pm25,pm10,no2,co,so2,o3,temperature,humidity
0,2025-02-19 01:30:00+05:30,127.0,171.0,24.5,1.60,17.3,22.0,14.2,94.0
1,2025-02-19 01:45:00+05:30,127.0,171.0,24.0,1.85,16.6,20.3,14.2,94.0
2,2025-02-19 02:00:00+05:30,127.0,NaN,23.2,1.72,17.6,20.6,14.3,95.0
3,2025-02-19 02:15:00+05:30,127.0,171.0,20.6,1.73,18.1,21.5,14.2,95.0
4,2025-02-19 02:30:00+05:30,108.0,135.0,18.0,1.74,18.1,20.9,14.1,95.0
5,2025-02-19 02:45:00+05:30,108.0,135.0,16.9,2.00,17.1,20.5,14.0,95.0
6,2025-02-19 03:00:00+05:30,108.0,NaN,15.9,1.76,16.7,20.2,13.9,95.0
7,2025-02-19 03:15:00+05:30,108.0,135.0,15.9,1.88,16.7,20.9,13.7,95.0
8,2025-02-19 03:30:00+05:30,96.0,115.0,16.1,1.80,16.8,21.1,13.7,95.0
9,2025-02-19 03:45:00+05:30,96.0,115.0,15.4,2.06,16.4,20.2,13.5,95.0


In [15]:
processed_folder = "processed_data"

os.makedirs(processed_folder, exist_ok=True)

feb_clean.to_csv(
    f"{processed_folder}/gwalior_air_quality_2025-02.csv",
    index=False
)

print("February combined dataset saved successfully!")

February combined dataset saved successfully!


In [16]:
# ==========================================
# COLLECT REMAINING MONTHS
# March 2025 → September 2026
# ==========================================

from datetime import datetime
from dateutil.relativedelta import relativedelta

start = datetime(2025, 3, 1)
end = datetime(2026, 9, 18)

current = start

all_monthly_data = {}

while current < end:

    month_start = current
    month_end = min(
        current + relativedelta(months=1),
        end
    )

    month_name = month_start.strftime("%Y-%m")

    start_str = month_start.strftime("%Y-%m-%dT%H:%M:%S+05:30")
    end_str = month_end.strftime("%Y-%m-%dT%H:%M:%S+05:30")

    print("\n" + "=" * 50)
    print(f"COLLECTING: {month_name}")
    print("=" * 50)

    all_monthly_data[month_name] = {}

    for parameter, sensor_id in usable_sensors.items():

        print(f"\n{parameter}...")

        readings = get_sensor_measurements_safe(
            sensor_id,
            start_str,
            end_str
        )

        all_monthly_data[month_name][parameter] = readings

        print(f"{parameter}: {len(readings)} readings")

    print(f"\n{month_name} completed!")

    current = month_end

print("\n==========================================")
print("ALL REMAINING MONTHS COLLECTED")
print("==========================================")


COLLECTING: 2025-03

pm25...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 733 readings
pm25: 2733 readings

pm10...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 539 readings
pm10: 2539 readings

no2...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 702 readings
no2: 2702 readings

co...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 630 readings
co: 2630 readings

so2...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 631 readings
so2: 2631 readings

o3...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 686 readings
o3: 2686 readings

temperature...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 779 readings
temperature: 2779 readings

humidity...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 779 readings
humidity: 2779 readings

2025-03 completed!

COLLECTING: 2025-04

pm25...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 207 readings
pm25: 2207 readings

pm10...
Page 1: 1000 readings
Page 2: 1000 readings
Page 3: 146 readings
pm10: 21

In [17]:
# Check collected months and parameters

print("Months collected:")
print(list(all_monthly_data.keys()))

print("\nNumber of parameters per month:")
for month, data in all_monthly_data.items():
    print(month, "→", len(data), "parameters")

Months collected:
['2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']

Number of parameters per month:
2025-03 → 8 parameters
2025-04 → 8 parameters
2025-05 → 8 parameters
2025-06 → 8 parameters
2025-07 → 8 parameters
2025-08 → 8 parameters
2025-09 → 8 parameters
2025-10 → 8 parameters
2025-11 → 8 parameters
2025-12 → 8 parameters
2026-01 → 8 parameters
2026-02 → 8 parameters
2026-03 → 8 parameters
2026-04 → 8 parameters
2026-05 → 8 parameters
2026-06 → 8 parameters
2026-07 → 8 parameters
2026-08 → 8 parameters
2026-09 → 8 parameters


In [18]:
# ==========================================
# COMBINE AND SAVE ALL REMAINING MONTHS
# ==========================================

for month, month_data in all_monthly_data.items():

    print(f"\nProcessing {month}...")

    combined = None

    for parameter, readings in month_data.items():

        if not readings:
            continue

        df = pd.DataFrame(readings)

        # Extract local timestamp
        df["timestamp"] = df["period"].apply(
            lambda x: x["datetimeFrom"]["local"]
        )

        # Keep only timestamp and value
        df = df[["timestamp", "value"]].copy()

        # Rename value to parameter name
        df = df.rename(columns={"value": parameter})

        # Combine parameters using timestamp
        if combined is None:
            combined = df
        else:
            combined = combined.merge(
                df,
                on="timestamp",
                how="outer"
            )

    # Sort by timestamp
    combined["timestamp"] = pd.to_datetime(
        combined["timestamp"]
    )

    combined = combined.sort_values("timestamp")

    # Remove accidental duplicate timestamps
    combined = combined.drop_duplicates(
        subset=["timestamp"]
    )

    # Save
    output_file = (
        f"processed_data/"
        f"gwalior_air_quality_{month}.csv"
    )

    combined.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")
    print(f"Rows: {len(combined)}")
    print(f"Columns: {len(combined.columns)}")

print("\n==========================================")
print("ALL MONTHLY DATASETS SAVED")
print("==========================================")


Processing 2025-03...
Saved: processed_data/gwalior_air_quality_2025-03.csv
Rows: 2779
Columns: 9

Processing 2025-04...
Saved: processed_data/gwalior_air_quality_2025-04.csv
Rows: 2270
Columns: 9

Processing 2025-05...
Saved: processed_data/gwalior_air_quality_2025-05.csv
Rows: 2461
Columns: 9

Processing 2025-06...
Saved: processed_data/gwalior_air_quality_2025-06.csv
Rows: 2313
Columns: 9

Processing 2025-07...
Saved: processed_data/gwalior_air_quality_2025-07.csv
Rows: 2343
Columns: 9

Processing 2025-08...
Saved: processed_data/gwalior_air_quality_2025-08.csv
Rows: 2574
Columns: 9

Processing 2025-09...
Saved: processed_data/gwalior_air_quality_2025-09.csv
Rows: 2584
Columns: 9

Processing 2025-10...
Saved: processed_data/gwalior_air_quality_2025-10.csv
Rows: 2555
Columns: 9

Processing 2025-11...
Saved: processed_data/gwalior_air_quality_2025-11.csv
Rows: 2406
Columns: 9

Processing 2025-12...
Saved: processed_data/gwalior_air_quality_2025-12.csv
Rows: 2587
Columns: 9

Processin

In [19]:
# ==========================================
# CREATE MASTER HISTORICAL DATASET
# ==========================================

import glob

files = glob.glob(
    "processed_data/gwalior_air_quality_*.csv"
)

all_dfs = []

for file in files:

    df = pd.read_csv(file)

    all_dfs.append(df)

# Combine all months
master_df = pd.concat(
    all_dfs,
    ignore_index=True
)

# Convert timestamp
master_df["timestamp"] = pd.to_datetime(
    master_df["timestamp"]
)

# Sort chronologically
master_df = master_df.sort_values(
    "timestamp"
).reset_index(drop=True)

# Remove any duplicate timestamps
master_df = master_df.drop_duplicates(
    subset=["timestamp"]
).reset_index(drop=True)

# Save master dataset
master_file = (
    "processed_data/"
    "gwalior_air_quality_historical.csv"
)

master_df.to_csv(
    master_file,
    index=False
)

print("==========================================")
print("MASTER DATASET CREATED")
print("==========================================")

print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))

print("\nDate range:")
print(master_df["timestamp"].min())
print("to")
print(master_df["timestamp"].max())

print("\nColumns:")
print(list(master_df.columns))

print("\nSaved as:")
print(master_file)

MASTER DATASET CREATED
Rows: 45227
Columns: 9

Date range:
2025-02-19 01:30:00+05:30
to
2026-09-17 19:45:00+05:30

Columns:
['timestamp', 'pm25', 'pm10', 'no2', 'co', 'so2', 'o3', 'temperature', 'humidity']

Saved as:
processed_data/gwalior_air_quality_historical.csv


In [1]:
import os
import requests

url = "https://api.openaq.org/v3/locations/10907/latest"

headers = {
    "X-API-Key": os.getenv("OPENAQ_API_KEY")
}

response = requests.get(url, headers=headers, timeout=30)

print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()

    for item in data.get("results", []):
        print(
            item.get("sensorsId"),
            item.get("parameter", {}).get("name"),
            item.get("datetime", {}).get("utc"),
            item.get("value")
        )
else:
    print(response.text[:500])

Status: 200
12236152 None 2026-10-31T09:45:00Z 29.0
12236154 None 2026-10-31T09:45:00Z 51.0
12236156 None 2026-10-31T09:45:00Z 30.7
12236148 None 2026-10-31T09:45:00Z 1.91
12236151 None 2026-10-31T09:45:00Z 40.3
12236155 None 2026-10-31T09:45:00Z 3.0
12236149 None 2026-10-31T09:45:00Z 4.5
12236153 None 2026-10-31T09:45:00Z 2.0
12236150 None 2026-10-31T09:45:00Z 50.7
14340982 None 2026-10-31T09:45:00Z 0.3
14340981 None 2026-10-31T09:45:00Z 243.0
14340980 None 2026-10-31T09:45:00Z 0.0286
36584 None 2022-10-16T16:00:00Z 59.0
36593 None 2022-10-16T16:00:00Z 23.8
36580 None 2022-10-16T16:00:00Z 9.7
36554 None 2022-10-16T16:00:00Z 1310.0
36582 None 2022-10-16T16:00:00Z 132.0
36574 None 2022-10-16T16:00:00Z 50.4


In [2]:
import os
import requests
import pandas as pd

url = "https://api.openaq.org/v3/locations/10907/latest"

headers = {"X-API-Key": os.getenv("OPENAQ_API_KEY")}

data = requests.get(url, headers=headers, timeout=30).json()

sensor_map = {
    12236153: "PM2.5",
    12236152: "PM10",
    12236150: "NO2",
    12236148: "CO",
    12236155: "SO2",
    12236151: "O3",
    12236156: "Temperature",
    12236154: "Humidity"
}

for item in data.get("results", []):
    sid = item.get("sensorsId")
    if sid in sensor_map:
        dt = item.get("datetime", {}).get("utc")
        print(sensor_map[sid], ":", dt)

PM10 : 2026-10-31T09:45:00Z
Humidity : 2026-10-31T09:45:00Z
Temperature : 2026-10-31T09:45:00Z
CO : 2026-10-31T09:45:00Z
O3 : 2026-10-31T09:45:00Z
SO2 : 2026-10-31T09:45:00Z
PM2.5 : 2026-10-31T09:45:00Z
NO2 : 2026-10-31T09:45:00Z


In [4]:
import os
import requests
import pandas as pd

url = "https://api.openaq.org/v3/locations/10907/latest"

headers = {
    "X-API-Key": os.getenv("OPENAQ_API_KEY")
}

sensor_map = {
    12236153: "PM2.5",
    12236152: "PM10",
    12236150: "NO2",
    12236148: "CO",
    12236155: "SO2",
    12236151: "O3",
    12236156: "Temperature",
    12236154: "Humidity"
}

response = requests.get(url, headers=headers, timeout=30)

print("API Status:", response.status_code)

response.raise_for_status()

results = response.json().get("results", [])

rows = []

for item in results:
    sensor_id = item.get("sensorsId")

    if sensor_id not in sensor_map:
        continue

    dt = item.get("datetime")

    if isinstance(dt, dict):
        dt = dt.get("utc") or dt.get("local")

    if dt is None:
        continue

    timestamp_utc = pd.to_datetime(dt, utc=True)

    rows.append({
        "Pollutant": sensor_map[sensor_id],
        "Sensor ID": sensor_id,
        "Value": item.get("value"),
        "Timestamp UTC": timestamp_utc,
        "Timestamp IST": timestamp_utc.tz_convert("Asia/Kolkata")
    })

df = pd.DataFrame(rows)

if not df.empty:
    print("\nOPENAQ LATEST SENSOR READINGS\n")
    print(df.sort_values("Pollutant").to_string(index=False))

    print("\nNewest API timestamp (IST):",
          df["Timestamp IST"].max())
else:
    print("No matching sensor readings found.")

API Status: 200

OPENAQ LATEST SENSOR READINGS

  Pollutant  Sensor ID  Value             Timestamp UTC             Timestamp IST
         CO   12236148   1.91 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
   Humidity   12236154  51.00 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
        NO2   12236150  50.70 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
         O3   12236151  40.30 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
       PM10   12236152  29.00 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
      PM2.5   12236153   2.00 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
        SO2   12236155   3.00 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30
Temperature   12236156  30.70 2026-10-31 09:45:00+00:00 2026-10-31 15:15:00+05:30

Newest API timestamp (IST): 2026-10-31 15:15:00+05:30
